## This Document is used to obtain data from FPBase

In the database certain protein entries are not complete. This file seeks to pull the whole database and filter out proteins that don't have complete entries. Following this, the database is manually checked to see which proteins have experimentally determined structures available in the protein database (PDB).

In [19]:
import requests
import pandas as pd

FPBASE_URL = "https://www.fpbase.org/graphql/"

API Helper function

In [20]:
def run_query(query: str) -> dict:
    response = requests.post(
        FPBASE_URL,
        json={"query": query},
        headers={"Content-Type": "application/json"},
    )

    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise Exception(data["errors"])

    return data["data"]

Fetch Data

In [21]:
query = """
{
  proteins {
    name
    slug
    seq
    agg
    pdb
    cofactor
    defaultState {
      exMax
      emMax
      extCoeff
      qy
      brightness
    }
  }
}
"""

data = run_query(query)
proteins = data["proteins"]

Build Dataframe

In [22]:
rows = []

for p in proteins:
    state = p.get("defaultState") or {}

    rows.append({
        "name": p.get("name"),
        "fpbase_id": p.get("slug"),
        "sequence": p.get("seq"),
        "aggregation_type": p.get("agg"),
        "excitation_wavelength": state.get("exMax"),
        "emission_wavelength": state.get("emMax"),
        "extinction_coefficient": state.get("extCoeff"),
        "quantum_yield": state.get("qy"),
        "brightness": state.get("brightness"),
        "pdb_codes": p.get("pdb"),
        "cofactor": p.get("cofactor"),
    })

df = pd.DataFrame(rows)

print(df.shape)
df.head()

(1040, 11)


,name,fpbase_id,sequence,aggregation_type,excitation_wavelength,emission_wavelength,extinction_coefficient,quantum_yield,brightness,pdb_codes,cofactor
0,10B,10b,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,NaN,513.0,525.0,30800.0,NaN,NaN,[],NaN
1,11,11,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,NaN,502.0,512.0,33000.0,NaN,NaN,[],NaN
2,22G,22g,MSVIKPDMKIKLRMEGAVNGHPFAIEGVGLGKPFEGKQSMDLKVKE...,T,NaN,NaN,NaN,NaN,NaN,[2Z6X],NaN
3,(3-F)Tyr-EGFP,3-ftyr-egfp,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,NaN,484.0,514.0,NaN,NaN,NaN,[1RRX],NaN
4,5B,5b,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,NaN,512.0,524.0,19400.0,NaN,NaN,[],NaN


Clean up dataframe

In [23]:
required_cols = [
    "brightness",
    "quantum_yield",
    "extinction_coefficient",
    "sequence",
    "emission_wavelength",
    "excitation_wavelength",
]

df_clean = df.dropna(subset=required_cols).copy()

print("Original:", len(df))
print("Remaining:", len(df_clean))
print("Removed:", len(df) - len(df_clean))

df_clean.isna().sum()

Original: 1040
Remaining: 592
Removed: 448


name                        0
fpbase_id                   0
sequence                    0
aggregation_type           72
excitation_wavelength       0
emission_wavelength         0
extinction_coefficient      0
quantum_yield               0
brightness                  0
pdb_codes                   0
cofactor                  541
dtype: int64

Clean up Dataframe

In [24]:
df = df_clean.copy()

df = df.rename(columns={
    "name": "Protein Name",
    "fpbase_id": "FPBase ID",
    "pdb_codes": "PDB code",
    "cofactor": "Chromophore/ligand",
    "sequence": "Protein sequence",
    "excitation_wavelength": "Excitation wavelength",
    "emission_wavelength": "Emission wavelength",
    "extinction_coefficient": "EC value",
    "quantum_yield": "QY value",
    "brightness": "Brightness",
})

# ONLY derived metric you actually need
df["Stokes shift"] = df["Emission wavelength"] - df["Excitation wavelength"]

# IMPORTANT: manual fields (leave blank)
df["Molecular weight (kDa)"] = ""
df["PDB code"] = ""
df["Chromophore/ligand"] = ""
df["Protein sequence"] = ""

Final Export

In [27]:
filtered_df = df[
    (df["Emission wavelength"] >= 500) &
    (df["Emission wavelength"] <= 600)
].copy()

desired_order = [
    "Protein Name",
    "PDB code",
    "Chromophore/ligand",
    "Protein sequence",
    "Excitation wavelength",
    "Emission wavelength",
    "Stokes shift",
    "EC value",
    "QY value",
    "Brightness",
    "Molecular weight (kDa)",
    "FPBase ID",
]

filtered_df = filtered_df[desired_order]

filtered_df.to_excel(
    "filtered_fluorescent_proteins_clean.xlsx",
    index=False
)

print("Saved:", len(filtered_df), "rows")

Saved: 363 rows
